# Notebook 4 : WPA online & VFAST (Laplace et gaussien)

In [2]:
%reload_ext autoreload
%autoreload 2
import sys, pathlib
ROOT = pathlib.Path.cwd().parent                 
if str(ROOT) not in sys.path:                    
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, SelectionSlider, Dropdown

import online_dp as dp
from online_dp.config import Config
from online_dp.cache import compute_or_load
from online_dp import data, metrics, basis, gam, mechanisms, vfast, viz

cfg = Config(
    data_dir=str(ROOT.parent / "data" / "DataDiffusionDeepCourboGen") + "/",   # adapte si besoin
    cache_dir=str(ROOT / "cache"),
    n_panel=500, n_target=50, seed=0,        
    clip_quantile=0.95,                    
)
dp.cache.CACHE = pathlib.Path(cfg.cache_dir)       # cache partage par les 4 notebooks
cfg

Config(data_dir='/home/G70186/data/DataDiffusionDeepCourboGen/', cache_dir='/home/G70186/onlinedp_stage26/cache', N=1000, n_panel=500, n_target=50, seed=0, calendar_start='2022-10-02 20:00:00', slots_per_day=48, pmax=47, n_groups=100, fit_subsample=10000, nmf_max_iter=500, nmf_tol=0.0001, clip_quantile=0.95, delta_dp=1e-05)

In [3]:
D = compute_or_load(cfg.key('dataset'), lambda: data.build_dataset(cfg))
df_daily, df_agg = D['df_daily'], D['df_agg']
panel_profiles, panel_tensor, panel_ids = D['panel_profiles'], D['panel_tensor'], D['panel_ids']
target_users = D['target_users']
print('panel', panel_tensor.shape, '| agrege cible', df_agg.shape, '| cible', len(target_users))

[cache] 'dataset__N1000_np500_nt50_s0' rechargé (joblib).
panel (500, 362, 48) | agrege cible (362, 48) | cible 50


## Base de déploiement (SVD matchée à n_target, p* = NB 3) et sensibilités

`Delta / Delta_mean` : sensibilité **L1 par coordonnée** (mécanisme de Laplace, allocation ε/p).
`C / Delta2` : sensibilité **L2 scalaire** (mécanisme gaussien, pas de split par coordonnée).
`alpha_clip` est l'objet réellement perturbé (coefficients de la cible, clippés puis agrégés par jour).

In [4]:
P_RUN = 6
W = basis.build_matched_basis(panel_tensor, cfg.n_target, cfg,
                              p_list=[P_RUN], methods=('svd',))['svd'][P_RUN]

Delta, Delta_mean = basis.panel_sensitivity(panel_profiles, W, cfg.n_target, cfg.clip_quantile)     # L1
C, Delta2         = basis.panel_sensitivity_l2(panel_profiles, W, cfg.n_target, cfg.clip_quantile)  # L2

alpha_clip = mechanisms.clip_aggregate_coeffs(df_daily.loc[list(target_users)], W, Delta)
L_t = df_agg.values
T   = L_t.shape[0]
print(f"p = {W.shape[1]} | T = {T} jours")
print('Delta_mean (L1, par coord) :', np.round(Delta_mean, 3))
print(f"Delta2 (L2 scalaire)       : {float(Delta2):.3f}")

p = 6 | T = 362 jours
Delta_mean (L1, par coord) : [0.244 0.073 0.071 0.05  0.058 0.051]
Delta2 (L2 scalaire)       : 0.273


## WPA online

In [5]:
eps_grid = np.arange(1, 101, 2)

# --- WPA Laplace ---
L_lap_seq = mechanisms.wpa_laplace_grid(alpha_clip, W, Delta_mean, eps_grid, mode='sequential')  # eps/T
L_lap_day = mechanisms.wpa_laplace_grid(alpha_clip, W, Delta_mean, eps_grid, mode='per_day')      # eps/jour
# --- WPA gaussien (calibration RDP) ---
L_gau_seq = mechanisms.wpa_gaussian_grid(alpha_clip, W, Delta2, eps_grid, cfg.delta_dp, mode='sequential')
L_gau_day = mechanisms.wpa_gaussian_grid(alpha_clip, W, Delta2, eps_grid, cfg.delta_dp, mode='per_day')

def _eps_curve(L_grid):
    """L_grid (n_eps, T, 48) -> dict metric -> {mean, lo, hi} sur les jours, par eps."""
    out = {m: {'mean': [], 'lo': [], 'hi': []} for m in metrics.METRICS}
    for i in range(L_grid.shape[0]):
        md = metrics.per_day_metrics(L_t, L_grid[i])
        for m in metrics.METRICS:
            s = metrics.stats_over_axis(md[m], axis=0, q_lo=0.05, q_hi=0.95)
            out[m]['mean'].append(s['mean']); out[m]['lo'].append(s['lo']); out[m]['hi'].append(s['hi'])
    return {m: {k: np.array(v) for k, v in d.items()} for m, d in out.items()}

wpa = {('laplace', 'sequential'): _eps_curve(L_lap_seq),
       ('laplace', 'per_day'):    _eps_curve(L_lap_day),
       ('gaussian', 'sequential'):_eps_curve(L_gau_seq),
       ('gaussian', 'per_day'):   _eps_curve(L_gau_day)}
print('grilles WPA prêtes :', {k: v['RMSE']['mean'].shape for k, v in wpa.items()})

grilles WPA prêtes : {('laplace', 'sequential'): (50,), ('laplace', 'per_day'): (50,), ('gaussian', 'sequential'): (50,), ('gaussian', 'per_day'): (50,)}


## VFAST : tuning de $q$     ($Q = q \times I_p$) sur agrégats publics et run 

## Réglage de $q$ : courbes de tuning sur agrégats publics

In [6]:
EPS_FAST = 20.0
q_grid   = np.linspace(0, 1, 500)

rng = np.random.default_rng(123)
eval_aggs = [panel_tensor[rng.choice(len(panel_ids), size=cfg.n_target, replace=False)].mean(0)
             for _ in range(15)]                       # agregats publics DP-safe

cfg_fast_lap = vfast.FastConfig(M_ratio=0.15, P0=1e6, theta=10.0, xi=0.1, Cp=0.9, Ci=0.1, Cd=0.0,
                                mechanism='laplace',  delta=cfg.delta_dp)
cfg_fast_gau = vfast.FastConfig(M_ratio=0.15, P0=1e6, theta=10.0, xi=0.1, Cp=0.9, Ci=0.1, Cd=0.0,
                                mechanism='gaussian', delta=cfg.delta_dp)

store_q_lap, qbest_lap = compute_or_load(
    cfg.key('qsearch', p=P_RUN, mech='laplace'),
    lambda: vfast.grid_search_q(eval_aggs, W, Delta_mean, EPS_FAST, cfg_fast_lap, q_grid))
store_q_gau, qbest_gau = compute_or_load(
    cfg.key('qsearch', p=P_RUN, mech='gaussian'),
    lambda: vfast.grid_search_q(eval_aggs, W, Delta_mean, EPS_FAST, cfg_fast_gau, q_grid, Delta2=Delta2))

Q_lap = qbest_lap['RMSE'] * np.eye(P_RUN)
Q_gau = qbest_gau['RMSE'] * np.eye(P_RUN)

L_vf_lap, a_vf_lap, diag_lap = vfast.fast_vectorial(alpha_clip.values, W, Delta_mean, EPS_FAST,
                                                    cfg_fast_lap, Q_matrix=Q_lap)
L_vf_gau, a_vf_gau, diag_gau = vfast.fast_vectorial(alpha_clip.values, W, Delta_mean, EPS_FAST,
                                                    cfg_fast_gau, Q_matrix=Q_gau, Delta2=Delta2)
print(f"q* Laplace = {qbest_lap['RMSE']:.3f} | q* gaussien = {qbest_gau['RMSE']:.3f}")
print(f"publications : Laplace {diag_lap['n_samples']}/{diag_lap['M']}"
      f" | gaussien {diag_gau['n_samples']}/{diag_gau['M']} (sigma={diag_gau['sigma']:.3f})")

[cache] 'qsearch__N1000_np500_nt50_s0_p6_mechlaplace' rechargé (joblib).
[cache] 'qsearch__N1000_np500_nt50_s0_p6_mechgaussian' rechargé (joblib).
q* Laplace = 0.024 | q* gaussien = 0.048
publications : Laplace 22/55 | gaussien 23/55 (sigma=0.644)


In [ ]:
@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in store_q_lap],
                          value='RMSE', description='metrique',
                          layout=dict(width='300px'), style={'description_width': '80px'}))
def show_qtuning(metric):
    fig = go.Figure()
   
    yvals = np.concatenate([store_q_lap[metric][k] for k in ('q25', 'q75', 'mean')]
                           + [store_q_gau[metric][k] for k in ('q25', 'q75', 'mean')])
    ymin, ymax = float(yvals.min()), float(yvals.max())
    for store, qbest, dash, lab in [(store_q_lap, qbest_lap, 'solid', 'Laplace'),
                                    (store_q_gau, qbest_gau, 'dash',  'gaussien')]:
        s = store[metric]; grp = f"vfast-{lab}"; q = qbest[metric]
        fig.add_trace(viz.band(q_grid, s['q75'], s['q25'],
                               viz.rgba(viz.COLOR_VFAST, 0.08)).update(legendgroup=grp))
        fig.add_trace(go.Scatter(x=q_grid, y=s['mean'], mode='lines', name=f"VFAST {lab}",
                      legendgroup=grp, line=dict(color=viz.COLOR_VFAST, width=2.4, dash=dash)))
        fig.add_trace(go.Scatter(x=[q, q], y=[ymin, ymax], mode='lines+text', legendgroup=grp,
                      showlegend=False, line=dict(color=viz.COLOR_VFAST, width=1.2, dash=dash),
                      text=['', f"q* {lab} = {q:.2f}"], textposition='top center',
                      textfont=dict(color=viz.COLOR_VFAST, size=11), hoverinfo='skip'))
    fig.update_xaxes(title_text='q')
    fig.update_yaxes(title_text=metrics.METRIC_LABEL[metric])
    viz.apply_default_layout(
        fig, title=f"Reglage de q sur agregats publics — {metrics.METRIC_LABEL[metric]}",
        height=520, width=1000, legend_pos='top')
    fig.update_layout(legend=dict(groupclick='togglegroup'))
    fig.show()

In [ ]:
# Courbes VFAST utilite-vs-eps (q* fixe)
def _vfast_eps_curve(config, Q, Delta2=None):
    out = {m: {'mean': [], 'lo': [], 'hi': []} for m in metrics.METRICS}
    for e in eps_grid:
        L_vf = vfast.fast_vectorial(alpha_clip.values, W, Delta_mean, float(e), config,
                                    Q_matrix=Q, Delta2=Delta2)[0]
        md = metrics.per_day_metrics(L_t, L_vf)
        for m in metrics.METRICS:
            s = metrics.stats_over_axis(md[m], axis=0, q_lo=0.05, q_hi=0.95)
            out[m]['mean'].append(s['mean']); out[m]['lo'].append(s['lo']); out[m]['hi'].append(s['hi'])
    return {m: {k: np.array(v) for k, v in d.items()} for m, d in out.items()}

cfg_fast_aniso = vfast.FastConfig(
    M_ratio=0.15, P0=1e6, theta=10.0, xi=0.1,
    Cp=0.9, Ci=0.1, Cd=0.0,
    mechanism='gaussian_aniso',
    delta=cfg.delta_dp
)

vf = {
    'laplace':        _vfast_eps_curve(cfg_fast_lap,   Q_lap),
    'gaussian':       _vfast_eps_curve(cfg_fast_gau,   Q_gau,  Delta2=Delta2),
    'gaussian_aniso': _vfast_eps_curve(cfg_fast_aniso, Q_gau),  # Q_gau réutilisé, Delta2=None
}
print('courbes VFAST pretes.')

## Benchmark utilité / confidentialité : WPA vs VFAST

In [ ]:
#  Calcul des grilles WPA anisotrope 
L_gau_aniso_seq = mechanisms.wpa_gaussian_anisotropic_grid(
    alpha_clip, W, Delta_mean, eps_grid, cfg.delta_dp, mode='sequential')
L_gau_aniso_day = mechanisms.wpa_gaussian_anisotropic_grid(
    alpha_clip, W, Delta_mean, eps_grid, cfg.delta_dp, mode='per_day')

wpa[('gaussian_aniso', 'sequential')] = _eps_curve(L_gau_aniso_seq)
wpa[('gaussian_aniso', 'per_day')]    = _eps_curve(L_gau_aniso_day)

WPA_KEYS = [('laplace',        'sequential'), ('laplace',        'per_day'),
            ('gaussian',       'sequential'), ('gaussian',       'per_day'),
            ('gaussian_aniso', 'sequential'), ('gaussian_aniso', 'per_day')]

COLOR_ANISO = '#8e44ad'   # violet 

# Plancher : erreur de projection
floor_md = metrics.per_day_metrics(L_t, (L_t @ W) @ W.T)

@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in metrics.METRICS],
                          value='RMSE', description='metrique',
                          layout=dict(width='300px'), style={'description_width': '80px'}))
def show_benchmark(metric):
    fig = go.Figure()

    def add_curve(x, mean, hi, lo, color, dash, name, grp, fill_alpha):
        fig.add_trace(viz.band(x, hi, lo, viz.rgba(color, fill_alpha)).update(legendgroup=grp))
        fig.add_trace(go.Scatter(x=x, y=mean, mode='lines', name=name, legendgroup=grp,
                                 line=dict(color=color, width=2.0, dash=dash)))

    #  Plancher projection 
    sf  = metrics.stats_over_axis(floor_md[metric], axis=0, q_lo=0.05, q_hi=0.95)
    add_curve(eps_grid, np.full(eps_grid.shape, sf['mean']),
              np.full(eps_grid.shape, sf['hi']), np.full(eps_grid.shape, sf['lo']),
              viz.COLOR_PROJ, 'dot', f"plancher projection (p={P_RUN}, SVD)", 'floor', 0.10)

    #  WPA : Laplace + gaussien isotrope + gaussien anisotrope 
    mech_color = {**viz.MECH_COLOR, 'gaussian_aniso': COLOR_ANISO}
    mech_label = {'laplace': 'Laplace', 'gaussian': 'gaussien iso.', 'gaussian_aniso': 'gaussien aniso.'}

    for mech, mode in WPA_KEYS:
        s = wpa[(mech, mode)][metric]
        add_curve(eps_grid, s['mean'], s['hi'], s['lo'],
                  mech_color[mech], viz.MODE_DASH[mode],
                  f"WPA {mech_label[mech]} ({viz.MODE_LABEL[mode]})",
                  f"wpa-{mech}-{mode}", 0.05)

    #  VFAST : Laplace + gaussien isotrope + gaussien anisotrope
    vfast_styles = [
        ('laplace',        viz.COLOR_VFAST, 'solid',   'VFAST Laplace'),
        ('gaussian',       viz.COLOR_VFAST, 'dash',    'VFAST gaussien iso.'),
        ('gaussian_aniso', COLOR_ANISO,     'dashdot', 'VFAST gaussien aniso.'),
    ]
    for mech, color, dash, label in vfast_styles:
        s = vf[mech][metric]
        add_curve(eps_grid, s['mean'], s['hi'], s['lo'],
                  color, dash, label, f"vfast-{mech}", 0.06)

    fig.update_yaxes(type='log', title_text=f"{metrics.METRIC_LABEL[metric]} (échelle log)")
    fig.update_xaxes(title_text='epsilon (budget)')
    viz.apply_default_layout(
        fig, title=f"Utilité vs epsilon — WPA & VFAST (p={P_RUN}, n_target={cfg.n_target})",
        height=580, width=1050, legend_pos='top')
    fig.update_layout(legend=dict(orientation='v', x=0.99, y=0.99, xanchor='right', yanchor='top',
                                  groupclick='togglegroup',
                                  bgcolor='rgba(255,255,255,0.85)', bordercolor='#d5dbdb', borderwidth=1))
    fig.show()

## Reconstruction d'une journée

In [ ]:
@interact(eps=SelectionSlider(options=[int(e) for e in eps_grid], value=15, description='epsilon',
                              continuous_update=False, layout=dict(width='470px'),
                              style={'description_width': '70px'}),
          day=IntSlider(min=0, max=T - 1, value=T // 2, description='jour',
                        continuous_update=False, layout=dict(width='470px'),
                        style={'description_width': '70px'}))
def show_reconstruction(eps, day):
    ie = int(np.argmin(np.abs(eps_grid - eps)))
    Lvf_l = vfast.fast_vectorial(alpha_clip.values, W, Delta_mean, float(eps),
                                 cfg_fast_lap, Q_matrix=Q_lap)[0]
    Lvf_g = vfast.fast_vectorial(alpha_clip.values, W, Delta_mean, float(eps),
                                 cfg_fast_gau, Q_matrix=Q_gau, Delta2=Delta2)[0]
    slots = np.arange(48)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=slots, y=L_t[day], mode='lines', name='agrege reel',
                  line=dict(color=viz.COLOR_REAL, width=3)))
    # WPA Laplace (rouge) : eps/T plein, eps/jour pointille
    fig.add_trace(go.Scatter(x=slots, y=L_lap_seq[ie, day], mode='lines', name='WPA Laplace (eps/T)',
                  line=dict(color=viz.COLOR_LAPLACE, width=1.5, dash=viz.MODE_DASH['sequential'])))
    fig.add_trace(go.Scatter(x=slots, y=L_lap_day[ie, day], mode='lines', name='WPA Laplace (eps/jour)',
                  line=dict(color=viz.COLOR_LAPLACE, width=1.5, dash=viz.MODE_DASH['per_day'])))
    # WPA gaussien (bleu) : eps/T plein, eps/jour pointille
    fig.add_trace(go.Scatter(x=slots, y=L_gau_seq[ie, day], mode='lines', name='WPA gaussien (eps/T)',
                  line=dict(color=viz.COLOR_GAUSSIAN, width=1.5, dash=viz.MODE_DASH['sequential'])))
    fig.add_trace(go.Scatter(x=slots, y=L_gau_day[ie, day], mode='lines', name='WPA gaussien (eps/jour)',
                  line=dict(color=viz.COLOR_GAUSSIAN, width=1.5, dash=viz.MODE_DASH['per_day'])))
    # VFAST (violet) : Laplace plein, gaussien pointille
    fig.add_trace(go.Scatter(x=slots, y=Lvf_l[day], mode='lines', name='VFAST Laplace',
                  line=dict(color=viz.COLOR_VFAST, width=1.9)))
    fig.add_trace(go.Scatter(x=slots, y=Lvf_g[day], mode='lines', name='VFAST gaussien',
                  line=dict(color=viz.COLOR_VFAST, width=1.9, dash='dash')))
    viz.apply_default_layout(
        fig, title=f"Reconstruction — {viz.fr_date(df_agg.index[day])} — epsilon = {eps}",
        height=480, width=1000, legend_pos='top')
    fig.update_layout(legend=dict(orientation='v', x=0.01, y=0.99, xanchor='left', yanchor='top',
                                  bgcolor='rgba(255,255,255,0.85)', bordercolor='#d5dbdb', borderwidth=1))
    fig.update_xaxes(title_text='creneau demi-horaire')
    fig.update_yaxes(title_text='charge agregee', rangemode='tozero')
    fig.show()

## Sensibilité de VFAST aux hyperparamètres ($M_{\text{ratio}}$, $\theta$, $\xi$)

In [ ]:
PARAM_GRID = {'M_ratio': np.linspace(0.05, 0.50, 10),
              'theta':   np.linspace(2.0, 30.0, 10),
              'xi':      np.linspace(0.02, 0.40, 10)}

@interact(param=Dropdown(options=list(PARAM_GRID), value='M_ratio', description='hyperparam',
                         layout=dict(width='280px'), style={'description_width': '90px'}),
          mech=Dropdown(options=[('Laplace', 'laplace'), ('gaussien', 'gaussian')], value='laplace',
                        description='mecanisme', layout=dict(width='280px'),
                        style={'description_width': '90px'}),
          metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in metrics.METRICS],
                          value='RMSE', description='metrique',
                          layout=dict(width='280px'), style={'description_width': '90px'}))
def show_sensitivity(param, mech, metric):
    cfgf = cfg_fast_lap if mech == 'laplace' else cfg_fast_gau
    Qm   = Q_lap if mech == 'laplace' else Q_gau
    d2   = None if mech == 'laplace' else Delta2
    rows = vfast.sensitivity_1d(param, PARAM_GRID[param], alpha_clip.values, W, Delta_mean, Qm,
                                cfgf, L_true=L_t, eps=EPS_FAST, metric=metric, Delta2=d2)
    xs  = [r[param] for r in rows]
    err = [r['error'] for r in rows]
    ns  = [r['n_samples'] for r in rows]
    lab = metrics.METRIC_LABEL[metric]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=xs, y=err, mode='lines+markers', name=lab,
                  line=dict(color=viz.COLOR_VFAST, width=2.4), marker=dict(size=7)),
                  secondary_y=False)
    fig.add_trace(go.Scatter(x=xs, y=ns, mode='lines+markers', name='# publications',
                  line=dict(color=viz.COLOR_REAL, width=1.8, dash='dot'), marker=dict(size=6)),
                  secondary_y=True)
    fig.update_yaxes(title_text=lab, secondary_y=False)
    fig.update_yaxes(title_text='# publications (n_samples)', secondary_y=True)
    fig.update_xaxes(title_text=param)
    fig.update_layout(
        title=dict(text=f"Sensibilite VFAST {mech} a {param} — {lab}  (eps = {EPS_FAST})",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        height=500, width=1000, template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
    fig.show()